<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Environment Setup

In [1]:
pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass()

In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

In [4]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


## Understanding the Warehouse

Before I can finalize the data contract, I need to understand what the warehouse actually contains.

At Week 3, I am still testing the feasibility of the project I framed in Weeks 1–2. I want to confirm the daily table's grain, choose a middle development period, inspect measurement availability, and determine whether I can separate historical information from a later outcome without leakage.


In [6]:
clients = con.sql(f"""
    SELECT
        client_hash_id,
        access_profile,
        gsc_data_start,
        ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print(
    "clients with 12+ months of GSC history:",
    (
        clients["gsc_data_start"]
        <= clients["gsc_data_start"].dropna().max()
        - __import__("pandas").Timedelta(days=365)
    ).sum()
)

clients.head(10)

clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


>The source table is a daily fact table, so one source row should represent one reporting date for one pseudonymized content page and client. I will verify the expected grain `report_date + client_hash_id + content_hash_id` rather than assuming it.
>
>For the first page-level modeling frame, I plan to aggregate those daily rows into **one row per pseudonymized page** using two adjacent 30-day periods around a March 2026 development endpoint. The earlier period will provide historical measurements and the later period will provide an observed performance outcome.
>
>This is my first workable window design, not necessarily the final capstone specification. If later validation shows that a longer history or a different eligibility rule is needed, I can revise the modeling contract then.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

>For this first warehouse specification, the historical feature candidates are `imp_prev30`, `clk_prev30`, and `pos_prev30`.
>
>The later-period impressions field `imp_last30` will be used to construct a decline label, so it is a **label ingredient**, not a predictive feature.
>
>`client_hash_id` and `content_hash_id` are **context fields**. I need them for page identity, grouping, inspection, and later validation, but I should not use the encoded IDs themselves as model inputs.
>
>I will keep `clk_last30` and `pos_last30` available for inspection, but I classify them as **excluded outcome-period fields** for an honest prediction model because they are measured after the intended decision point.
>
>I also plan to exclude any existing product decision scores or flags if I encounter them, because learning from an existing decision would not answer whether observable historical signals contain useful information.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

> Rather than relying solely on the warehouse documentation, I verified each part of the data contract using SQL queries against the selected March 2026 partition.

>I first performed a grain verification by grouping the data using report_date, client_hash_id, and content_hash_id, then checking whether any combination appeared more than once.
Since the query returned no duplicate rows, it confirmed that every record represents one content page for one client on one reporting date, matching the intended unit of analysis.

>I then verified the selected slice by checking both the row count and reporting date range for the partition.
The March 2026 partition contained 9,841,378 rows, and the minimum and maximum reporting dates confirmed that the extracted data matched the intended monthly window used throughout the notebook.
Finally, I verified data availability by filtering the dataset with ga4_data_available IS TRUE, which returned 413,966 rows containing usable GA4 engagement metrics.
This shows that engagement-related analyses can only be performed on a subset of the warehouse and that the availability flag should always be considered before selecting GA4-based features.
Performing these verification queries before feature engineering helped confirm that the assumptions made in the data contract matched the actual warehouse rather than relying on documentation alone, which is especially important when working with large, partitioned datasets.*

## Verifying the Data Contract

Before building any features, I verified the assumptions made in my data contract using SQL queries. Rather than relying solely on the warehouse documentation, these queries confirm that the selected data follows the intended grain, covers the expected reporting window, and contains the necessary search and engagement signals for my analysis.

In [7]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicates
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicates


### Row Count and Reporting Window

After confirming the grain, I verified the size of the selected partition together with its reporting date range to ensure that the extracted data matches the intended March 2026 time window.

In [8]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### GA4 Availability

Finally, I verified how many records contain usable Google Analytics data by filtering with `ga4_data_available IS TRUE`. Since engagement metrics are not available for every row in the warehouse, this check helps determine the portion of data that can be used for engagement-related analyses.

In [9]:
con.sql(f"""
SELECT
    COUNT(*) AS ga4_available_rows
FROM {TABLES['fact_daily']}
WHERE
    month='2026-03'
    AND ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_available_rows
0,413966


## Verification Summary

**After Grain Verification**

If the duplicate-grain query is empty, the March daily fact behaves as expected at `report_date × client × content` grain. If it is not empty, I need to resolve that before aggregating.

**After Row Count**

The March count and min/max dates confirm that I am actually working in the intended middle development period rather than accidentally using the warehouse's latest month.

**After GA4 Availability**

The GA4 availability result tells me whether engagement signals can be treated as generally available. If availability is incomplete, I should not blindly interpret zero-filled or missing analytics values as genuine zero engagement.


## Building the Feature Frame

I now build the first page-level frame from the warehouse.

The goal at Week 3 is to verify that I can construct a clean historical-versus-later comparison:

```text
previous 30 days → decision point → most recent 30 days
```

I retain some later-period fields in the dataframe because they are useful for defining the outcome and demonstrating leakage. Their presence in the dataframe does **not** mean they are valid model features.


In [10]:
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        -- Impressions
        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_last30,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_last30,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_prev30,

        -- Position
        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_last30

    FROM {TABLES['fact_daily']} f,
         bounds b

    WHERE f.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING imp_prev30 >= 100
)

SELECT *
FROM windowed
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
features.head()

,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,clk_prev30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,2.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,0.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,2.0,7.904483


### Feature Availability

- **imp_prev30** – Knowable at the decision moment because it summarizes search impressions observed during the historical feature window before the prediction period.
- **imp_last30** – Included in the feature frame to construct the target label and demonstrate target leakage. It is intentionally excluded from the honest feature set because it belongs to the outcome window.
- **clk_prev30** – Knowable at the decision moment because it summarizes historical search clicks already observed before the prediction period.
- **clk_last30** – Included in the engineered feature frame to summarize recent search click activity. It is excluded from the honest feature set because it overlaps with the outcome period used during the leakage demonstration.
- **pos_last30** – Knowable at the decision moment because it represents the observed average search position during the selected feature window.

## Demonstrating Target Leakage

After establishing a leakage-safe feature set, I performed a simple leakage experiment to demonstrate how easily model evaluation can become misleading when information from the outcome window is accidentally included as a feature. The purpose of this experiment is not to improve performance, but to show why separating feature and target windows is necessary for honest model evaluation.

In [12]:
features["is_declining"] = (
    features["imp_last30"] < 0.8 * features["imp_prev30"]
).astype(int)

In [13]:
features[["imp_prev30", "imp_last30", "is_declining"]].head(10)

,imp_prev30,imp_last30,is_declining
0,251.0,219.0,0
1,4786.0,995.0,1
2,177.0,103.0,1
3,1518.0,736.0,1
4,335.0,269.0,0
5,339.0,794.0,0
6,10179.0,8601.0,0
7,282.0,417.0,0
8,926.0,895.0,0
9,430.0,604.0,0


### Honest Feature Set

I first defined a feature set that only contains information available before the prediction window.

In [14]:
honest_features = [
    "imp_prev30",
    "clk_prev30",
    "pos_last30"
]
print(honest_features)

['imp_prev30', 'clk_prev30', 'pos_last30']


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X = features[honest_features]
y = features["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("=== Honest Model ===")
print(classification_report(y_test, model.predict(X_test)))

=== Honest Model ===
              precision    recall  f1-score   support

           0       0.50      0.42      0.46      9925
           1       0.70      0.76      0.73     17887

    accuracy                           0.64     27812
   macro avg       0.60      0.59      0.59     27812
weighted avg       0.63      0.64      0.63     27812



### Introduce Leakage

To demonstrate target leakage, I intentionally added `imp_last30`, which directly contributes to the definition of the target label.

In [16]:
leaked_features = [
    "imp_prev30",
    "clk_prev30",
    "pos_last30",
    "imp_last30"
]

print(leaked_features)

['imp_prev30', 'clk_prev30', 'pos_last30', 'imp_last30']


In [17]:
X = features[leaked_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print("=== Leaked Model ===")
print(classification_report(y_test, model.predict(X_test)))

=== Leaked Model ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9925
           1       0.99      0.99      0.99     17887

    accuracy                           0.99     27812
   macro avg       0.99      0.99      0.99     27812
weighted avg       0.99      0.99      0.99     27812



### Observation

>*After intentionally adding imp_last30 to the feature set, the evaluation metrics increased compared to the honest model. While the higher score initially appears to indicate better predictive performance, the improvement is misleading because imp_last30 is directly involved in defining the target label. This allows the model to access information from the outcome window, demonstrating how target leakage can produce unrealistically optimistic evaluation results.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

> *The warehouse is an unbalanced panel, meaning clients have different amounts of historical data available depending on when their tracking began, so not every client contributes the same depth of information.
I also observed that some earlier records contain only Google Search Console data because Google Analytics tracking either started later or was unavailable for certain clients.
This means that engagement-based analyses require filtering with ga4_data_available IS TRUE instead of assuming missing values represent zero engagement.
Another limitation is the risk of target leakage when feature and outcome windows overlap. During the leakage experiment, I intentionally included imp_last30, which is directly involved in defining the target label, and observed how easily model performance could become artificially inflated.
Although the resulting scores appeared much better, they were not representative of a real prediction because the model had already been exposed to information from the outcome window.
Because of these limitations, I see this dataset as a strong foundation for supporting content prioritization and future performance prediction, but only when feature windows, target windows, and validation strategies are carefully separated to ensure that the results remain realistic and reproducible.*

## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.